# Tutorial: Production RAG Observability with Local Ollama

This notebook walks through the full project pipeline in small, runnable steps:

1. Configure an isolated tutorial workspace
2. Discover local Ollama models (`ollama list`)
3. Build the retrieval index
4. Run one grounded RAG request
5. Evaluate quality + latency + cost
6. Apply regression gate logic
7. Inspect stored metrics and traces

All outputs in this notebook are written under `artifacts/notebook_tutorial/` so existing project artifacts are not overwritten.


In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / "pyproject.toml").exists(), "Run notebook from project root"

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

NOTEBOOK_ARTIFACTS = PROJECT_ROOT / "artifacts" / "notebook_tutorial"
NOTEBOOK_ARTIFACTS.mkdir(parents=True, exist_ok=True)

print(f"project_root={PROJECT_ROOT}")
print(f"notebook_artifacts={NOTEBOOK_ARTIFACTS}")


project_root=/home/ahmad/AI/Projects/production-rag-ask-my-docs
notebook_artifacts=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial


## Step 1: Configure Isolated Runtime Paths

We set environment variables before loading settings so this notebook uses:
- `artifacts/notebook_tutorial/index`
- `artifacts/notebook_tutorial/observability/*`
- `artifacts/notebook_tutorial/*.json`

This keeps your existing artifacts untouched.


In [2]:
os.environ["ASK_MY_DOCS_INDEX_DIR"] = str(NOTEBOOK_ARTIFACTS / "index")
os.environ["ASK_MY_DOCS_TRACES_PATH"] = str(NOTEBOOK_ARTIFACTS / "observability" / "traces.jsonl")
os.environ["ASK_MY_DOCS_METRICS_DB_PATH"] = str(NOTEBOOK_ARTIFACTS / "observability" / "metrics.duckdb")
os.environ["ASK_MY_DOCS_ARTIFACTS_DIR"] = str(NOTEBOOK_ARTIFACTS)

from ask_my_docs.settings import get_settings

get_settings.cache_clear()
settings = get_settings()

print("settings loaded")
print(f"index_dir={settings.index_dir}")
print(f"traces_path={settings.traces_path}")
print(f"metrics_db_path={settings.metrics_db_path}")
print(f"thresholds_path={settings.thresholds_path}")


settings loaded
index_dir=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial/index
traces_path=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial/observability/traces.jsonl
metrics_db_path=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial/observability/metrics.duckdb
thresholds_path=configs/regression_thresholds.yaml


## Step 2: Discover Local Ollama Models

The project automatically chooses the first local model from `ollama list` unless you explicitly set:

`ASK_MY_DOCS_OLLAMA_MODEL=<model_name>`


In [3]:
from ask_my_docs.llm.ollama import parse_ollama_list, resolve_ollama_model

result = subprocess.run(["ollama", "list"], capture_output=True, text=True, check=True)
print(result.stdout)

models = parse_ollama_list(result.stdout)
selected_model = resolve_ollama_model(configured_model=settings.ollama_model, available_models=models)
print(f"selected_model={selected_model}")


NAME                       ID              SIZE      MODIFIED     
gemma4:31b-cloud           c382fbfbc73b    -         41 hours ago    
gemma4:12b                 4eb23ef187e2    7.6 GB    41 hours ago    
medgemma1.5:4b             433252621ab1    3.3 GB    41 hours ago    
medgemma:4b                9fe4e9a6c9bd    3.3 GB    42 hours ago    
functiongemma:270m         7c19b650567a    300 MB    42 hours ago    
nemotron-3-nano:4b         6cc467f05439    2.8 GB    42 hours ago    
granite4.1:3b              6fd349357287    2.1 GB    42 hours ago    
granite4.1:8b              444af1c4b2fe    5.3 GB    42 hours ago    
lfm2.5-thinking:1.2b       95bd9d45385f    731 MB    42 hours ago    
translategemma:4b          c49d986b0764    3.3 GB    42 hours ago    
translategemma:12b         c2f9a9ca1ec7    8.1 GB    42 hours ago    
minicpm-v4.6:1b            e95583acac77    1.6 GB    42 hours ago    
qwen3.5:9b                 6488c96fa5fa    6.6 GB    42 hours ago    
lfm2.5:8b              

## Step 3: Build the Hybrid Retrieval Index

This step:
- loads docs from `data/docs/*.md`
- chunks text
- builds BM25 + FAISS artifacts
- saves index to `settings.index_dir`


In [4]:
from ask_my_docs.retrieval import HybridRetriever, load_documents

# Clean tutorial index for reproducibility in this notebook run
if settings.index_dir.exists():
    shutil.rmtree(settings.index_dir)

documents = load_documents(settings.docs_dir)
retriever = HybridRetriever.from_documents(documents=documents, config=settings.retrieval)
retriever.save(settings.index_dir)

print(f"documents={len(documents)}")
print(f"chunks={retriever.chunk_count}")
print(f"index_saved_to={settings.index_dir}")


documents=5
chunks=5
index_saved_to=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial/index


## Step 4: Build the RAG Pipeline with Observability Hooks

The pipeline wires together:
- retriever
- Ollama generator
- OpenTelemetry tracer
- DuckDB metrics store
- token-cost estimator


In [5]:
from ask_my_docs.llm.ollama import OllamaGenerator
from ask_my_docs.observability.cost import CostCalculator, TokenPricing
from ask_my_docs.observability.metrics_store import MetricsStore
from ask_my_docs.observability.tracing import configure_tracing
from ask_my_docs.pipeline import RAGPipeline

tracer = configure_tracing(service_name=settings.service_name, trace_path=settings.traces_path)
metrics_store = MetricsStore(db_path=settings.metrics_db_path)
retriever = HybridRetriever.load(index_dir=settings.index_dir, config=settings.retrieval)
generator = OllamaGenerator(
    base_url=settings.ollama_base_url,
    timeout_seconds=settings.ollama_timeout_seconds,
    configured_model=settings.ollama_model,
)
cost_calculator = CostCalculator(
    TokenPricing(
        prompt_cost_per_1k_tokens=settings.pricing.prompt_cost_per_1k_tokens,
        completion_cost_per_1k_tokens=settings.pricing.completion_cost_per_1k_tokens,
    )
)

pipeline = RAGPipeline(
    retriever=retriever,
    tracer=tracer,
    metrics_store=metrics_store,
    cost_calculator=cost_calculator,
    generator=generator,
)

print(f"pipeline_ready=True")
print(f"ollama_model={generator.model}")


pipeline_ready=True


ollama_model=gemma4:12b


## Step 5: Ask One Question (Grounded Answer + Cost + Latency)

We run one request and save its full JSON payload.


In [6]:
from ask_my_docs.utils import write_json

question = "What is the SLA for invoice disputes?"
answer = pipeline.answer(question=question, top_k=settings.retrieval.top_k)

payload_path = NOTEBOOK_ARTIFACTS / "sample_answer.json"
write_json(payload_path, pipeline.to_payload(answer))

print(f"question={question}")
print(f"model={answer.model_name}")
print(f"answer={answer.answer}")
print(f"citations={answer.citations}")
print(f"latency_ms={answer.latency_ms}")
print(f"estimated_cost_usd={answer.estimated_cost_usd:.8f}")
print(f"payload_path={payload_path}")


question=What is the SLA for invoice disputes?
model=gemma4:12b
answer=All invoice disputes must be acknowledged within 1 business day [billing_disputes]. Standard-priority disputes have a target resolution SLA of 3 business days after acknowledgement, and Enterprise-priority disputes must be resolved within 24 hours [billing_disputes].
citations=['billing_disputes']
latency_ms=29088.981
estimated_cost_usd=0.00026820
payload_path=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial/sample_answer.json


## Step 6: Run Evaluation (Quality + Latency + Cost)

We evaluate on `data/eval/eval_set.jsonl` and persist a report.


In [7]:
from ask_my_docs.evaluation.runner import load_eval_examples, run_evaluation, save_eval_report

eval_examples = load_eval_examples(settings.eval_path)
report = run_evaluation(pipeline=pipeline, examples=eval_examples, top_k=settings.retrieval.top_k)

current_metrics_path = NOTEBOOK_ARTIFACTS / "current_metrics.json"
save_eval_report(current_metrics_path, report)

print(json.dumps(report.aggregate, indent=2))
print(f"current_metrics_path={current_metrics_path}")


{
  "num_examples": 5.0,
  "answer_f1_mean": 0.7193830352653882,
  "exact_match_mean": 0.0,
  "retrieval_recall_at_k_mean": 1.0,
  "latency_p50_ms": 21382.19,
  "latency_p95_ms": 32220.5438,
  "avg_cost_usd": 0.00026013,
  "avg_tokens": 717.2
}
current_metrics_path=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial/current_metrics.json


## Step 7: Apply Regression Gate

For tutorial reproducibility, we initialize a notebook-local baseline if missing,
then apply the configured gate policy.


In [8]:
from ask_my_docs.evaluation.gating import evaluate_gate, load_gate_config
from ask_my_docs.utils import read_json

baseline_path = NOTEBOOK_ARTIFACTS / "baseline_metrics.json"
if not baseline_path.exists():
    shutil.copyfile(current_metrics_path, baseline_path)
    print(f"baseline_initialized_from_current={baseline_path}")

baseline_payload = read_json(baseline_path)
baseline_aggregate_obj = baseline_payload.get("aggregate")
baseline_metrics = (
    {key: float(value) for key, value in baseline_aggregate_obj.items()}
    if isinstance(baseline_aggregate_obj, dict)
    else None
)

gate_config = load_gate_config(settings.thresholds_path)
gate_result = evaluate_gate(
    current_metrics={key: float(value) for key, value in report.aggregate.items()},
    baseline_metrics=baseline_metrics,
    config=gate_config,
)

print(f"gate_passed={gate_result.passed}")
if gate_result.failures:
    print("gate_failures:")
    for failure in gate_result.failures:
        print(f"- {failure}")
else:
    print("gate_failures=[]")


gate_passed=True
gate_failures=[]


## Step 8: Inspect Metrics Summary from DuckDB

This shows production-style aggregates over recorded requests.


In [9]:
summary = metrics_store.summarize()
print(json.dumps(summary, indent=2))


{
  "request_count": 12.0,
  "latency_p50_ms": 30647.819,
  "latency_p95_ms": 53514.43429999999,
  "avg_cost_usd": 0.000265925,
  "avg_retrieval_recall_at_k": 1.0,
  "avg_answer_f1": 0.7094320548732315,
  "avg_exact_match": 0.0
}


## Step 9: Inspect Trace Samples

Each request writes spans to `traces.jsonl`.


In [10]:
trace_path = settings.traces_path
print(f"trace_path={trace_path}")

if trace_path.exists():
    lines = trace_path.read_text(encoding="utf-8").splitlines()
    print(f"num_spans={len(lines)}")
    for raw in lines[:5]:
        span = json.loads(raw)
        print({
            "name": span.get("name"),
            "trace_id": span.get("trace_id"),
            "duration_ms": span.get("duration_ms"),
        })
else:
    print("No trace file found yet")


trace_path=/home/ahmad/AI/Projects/production-rag-ask-my-docs/artifacts/notebook_tutorial/observability/traces.jsonl
num_spans=34
{'name': 'rag.retrieve', 'trace_id': 'bf379dd30c555dfa360d759dc08e82c7', 'duration_ms': 0.541}
{'name': 'rag.generate', 'trace_id': 'bf379dd30c555dfa360d759dc08e82c7', 'duration_ms': 55934.695}
{'name': 'rag.request', 'trace_id': 'bf379dd30c555dfa360d759dc08e82c7', 'duration_ms': 55966.799}
{'name': 'rag.retrieve', 'trace_id': '64cd051c8399874ac81386b70f2dfd64', 'duration_ms': 0.469}
{'name': 'rag.generate', 'trace_id': '64cd051c8399874ac81386b70f2dfd64', 'duration_ms': 51532.726}


## Step 10: What to Try Next

- Change `ASK_MY_DOCS_OLLAMA_MODEL` and compare quality/latency/cost.
- Tighten threshold values in `configs/regression_thresholds.yaml`.
- Expand `data/eval/eval_set.jsonl` with harder domain-specific questions.
- Hook `artifacts/observability/metrics.duckdb` into your dashboard stack.
